# Whisper WebUI Persian - Colab

Runs the full WebUI (VAD, long audio, video, URLs, SRT/VTT/TXT/JSON) against the fine-tuned
Persian models, on a free Colab GPU.

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

Then run the cells in order.

## 1. Check the GPU

If this errors or prints nothing, the runtime is still on CPU - go back and switch it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. Check out the project

In [ ]:
REPO = "https://github.com/ashahdev403/Whisper-WebUi-Persian.git"

import os

if not os.path.isdir("Whisper-WebUi-Persian"):
    !git clone {REPO}

%cd /content/Whisper-WebUi-Persian
!git log --oneline -1
!ls

### Alternative: run from a local copy

If you have changes that are not pushed yet, zip the project folder locally, upload the zip with the
file browser on the left, and run this **instead** of the cell above:

```python
!unzip -q -o /content/Whisper-WebUi-Persian.zip -d /content/
%cd /content/Whisper-WebUi-Persian
```

## 3. Install dependencies

Deliberately **not** `pip install -r requirements.txt` - that would reinstall torch and can break
Colab's CUDA build. Colab already ships torch, torchaudio, numpy and ffmpeg, so only the rest is
installed here.

Takes about a minute. Ignore any pip dependency-resolver warnings about preinstalled Colab packages.

In [ ]:
!pip install -q "transformers>=4.48.0" accelerate gradio json5 ffmpeg-python yt-dlp more-itertools altair intervaltree srt

!ffmpeg -version | head -1

## 4. Sanity check

Confirms the merged config loads and the Persian models are registered, before spending time on a
model download.

In [ ]:
from src.config import ApplicationConfig

config = ApplicationConfig.parse_file("config.json5")

print("backend      :", config.whisper_implementation)
print("default model:", config.default_model_name)
print("language     :", config.language)
print("max duration :", config.input_audio_max_duration)
print("VAD          :", config.default_vad)
print()

for model in config.models:
    print(f"  {model.name:24s} {model.type:14s} {model.url}")

## 5. Quick CLI test

Faster to debug than the UI, and it proves the whole chain: model load -> VAD -> transcription ->
subtitle files.

Upload a Persian audio or video file with the file browser on the left, then set `AUDIO` to its path.
Leave it as-is to generate a synthetic test tone instead (which will transcribe to nothing - it only
checks that the pipeline runs).

In [ ]:
AUDIO = "/content/sample.wav"
MODEL = "Persian Small"   # or "Persian Large v3"

import os

if not os.path.exists(AUDIO):
    print("No file at", AUDIO, "- generating a 40s test tone instead.")
    !ffmpeg -y -loglevel error -f lavfi -i "sine=frequency=440:duration=40" -ar 16000 -ac 1 {AUDIO}

!python cli.py "{AUDIO}" --model "{MODEL}" --compute_type float16 --vad silero-vad --output_dir /content/out

In [ ]:
# Show what the CLI produced
import glob

for path in sorted(glob.glob("/content/out/*")):
    print("=" * 70)
    print(path)
    print("=" * 70)

    if path.endswith((".srt", ".txt", ".vtt")):
        with open(path, encoding="utf-8") as handle:
            print(handle.read()[:2000])

## 6. Launch the WebUI

This cell keeps running for as long as the server is up. Click the
`https://xxxxx.gradio.live` link that appears next to **Running on public URL**.

The first transcription downloads the model (~500 MB for Persian Small, ~3 GB for Persian Large v3),
so the first run takes a minute longer than the rest.

Stop the cell to shut the server down.

In [ ]:
!python app-shared.py --compute_type float16

## Notes

- **Public link.** `app-shared.py` creates a Gradio share link that anyone with the URL can use, and
  it stays up for 72 hours or until you stop the cell. Don't paste sensitive audio through it.
- **Speed.** `--compute_type float16` roughly halves memory and speeds up decoding on the GPU.
  Use `bfloat16` on an A100, and `float32` if you see NaNs.
- **Long files.** Keep VAD on `silero-vad`. There is no length limit in this notebook.
- **Sessions.** Colab reclaims idle runtimes. Runtime -> Manage sessions -> terminate when done,
  otherwise it eats your free compute.
- **Model cache.** Models land in `/root/.cache/huggingface` and are lost when the runtime resets.
  Mount Drive and set `HF_HOME` to a Drive path if you want them to persist.